# Deep dive: making OpenClaw traces readable as Phoenix sessions

Companion to [Section 1 of the cookbook](../COOKBOOK.md#1-trace-your-assistant).

**We kept one native trace per interaction and added a shared `session.id` for related turns.**
We also made the current question and response visible at the top of each trace.
Phoenix could then show a conversation as a sequence of turns, with the model and tool steps one click away.

This tutorial explains the code we actually deployed on September 28, 2026 (PDT):
OpenClaw **2026.9.5 (`ec9c1a1`)**, diagnostics exporter **2026.9.5**, and our observer **1.1.1**.
It covers the initial session-view patch and the later tool-result naming update.
It is a local customization, not a claim that every OpenClaw version needs the same patch.

**For:** developers who know basic Python and can read a little JavaScript. No tracing expertise required.

**You will learn to:**
1. Distinguish sessions, traces and spans.
2. Join the current request to the right run and group turns by the real conversation window.
3. Add readable root fields, tool names and available recorded reasoning.
4. Preserve native timing and avoid counting model tokens twice.
5. Verify a version-specific exporter patch and understand its limits.

Every executable example uses synthetic messages and the repository's actual helper functions.
Running all cells makes no model calls, sends no messages, uploads nothing and changes no installed exporter.
The saved outputs are teaching fixtures, not additional experiment results.

## 1. The problem: data was present, but the conversation was hard to follow

The initial Phoenix view mixed short gateway operations with agent work. The root of an interaction
did not show its current question and answer, and the records lacked the session field Phoenix needs
to group related interactions. Native model and tool spans already existed; we did not need to invent them.

| Level | Meaning | Example |
| --- | --- | --- |
| Session | A group of related turns in one actual conversation window | Ask a question, then ask a follow-up |
| Trace | One interaction | Answer the follow-up |
| Span | One operation inside that interaction | A model call or `memory_search` execution |

The resulting view has this shape. The names below illustrate the display; the exact number of calls varies.

```text
Phoenix session: one conversation window
  Trace A — Conversation: current question → response
    Model call
      Recorded reasoning (when available)
    Tool execution: arguments → result
    Model call → response
  Trace B — Conversation: follow-up question → response
    Model call → response

Separate Phoenix session: isolated heartbeat
  Trace C — Heartbeat: scheduled prompt → response or silence token
```

Grouping changes navigation. It does **not** change what conversation history the model receives.
The separate heartbeat `isolatedSession` intervention changes that behavior; this exporter patch does not.

## 2. Run this notebook locally

Open it from this repository and run from top to bottom with a Python 3 kernel, IPython, and Node.js
(Node 20 or newer is a straightforward choice for these examples). Python invokes the existing JavaScript
helpers in a separate local Node process for each example. No Phoenix credentials are needed.

The next cell locates the repository and checks the exact source snapshot used when this notebook was built.
If a file changes, review the tutorial against that version before updating its recorded hashes.

In [1]:
from pathlib import Path
import hashlib
import html
import importlib.util
import json
import shutil
import subprocess

from IPython.display import HTML, Markdown, display


def find_repository(start):
    """Find the checkout containing the observer source above the notebook directory."""
    for candidate in (start, *start.parents):
        if (candidate / 'observer/live-trace.js').is_file():
            return candidate
    raise RuntimeError('Open this notebook from inside the openclaw-evals checkout.')


REPO = find_repository(Path.cwd().resolve())
NODE = shutil.which('node')
if NODE is None:
    raise RuntimeError('Install Node.js and restart the kernel so node is on PATH.')
print('Found the repository and Node.js', subprocess.check_output([NODE, '--version'], text=True).strip())

Found the repository and Node.js v23.6.0


In [2]:
EXPECTED_SOURCE_HASHES = {
    "observer/index.js": "1cbd7789b028084a27e73e568bd3d97c410d25e327bf2ed2bb275a65e738e481",
    "observer/live-trace.js": "0444195a29da6ff9b718eb77b6c974432bdc25f131e53f103e81b3fd68cbd1fb",
    "observer/openclaw.plugin.json": "970f06cfef2df351367262ef1a367c264a6f417f30197de6c09d05bd30b0cb80",
    "observer/patch-native-exporter.py": "acccb9d5b8cf73cf84fe4d7bc6ae529e9f3cda9f2a7d23f8a5189f8618e80f9a",
    "observer/test-live-trace.js": "94bb5e5b3479ca2b722632ebbf4bb841cdcffe81ca5ca815ba62578a46911be6",
    "observer/test-observer.js": "edf8ddd744a2080039b7fa4f5724ddc61b6b10ab599b2ed90c921a1dc5a9c21f"
}

for relative_path, expected_hash in EXPECTED_SOURCE_HASHES.items():
    actual_hash = hashlib.sha256((REPO / relative_path).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f'Source changed: {relative_path}; review the tutorial first.'
print(f'All {len(EXPECTED_SOURCE_HASHES)} source snapshots match this tutorial.')

All 6 source snapshots match this tutorial.


### A small runner for the real JavaScript helpers

The Python wrapper below only imports the local modules and returns JSON. Importing `index.js` does not
register the plugin or start OpenClaw. The examples call the pure record-building and in-memory functions;
they do not call the local transcript writer. Each example gets a fresh process so hidden state from a
previous cell cannot change its result.

In [3]:
JS_IMPORTS = (
    'import assert from "node:assert/strict";\n'
    'import { configureLiveCapture, captureLiveRecord, buildSessionAttributes, '
    'captureModelOutput, buildRunAttributes, addToolResultNames, markAggregateUsage } from '
    + json.dumps((REPO / 'observer/live-trace.js').as_uri()) + ';\n'
    'import { buildRecord, redactText } from '
    + json.dumps((REPO / 'observer/index.js').as_uri()) + ';\n'
)
JS_FIXTURE = """
const options = {liveExport: true, sessionScope: "all", captureReasoning: "text"};
configureLiveCapture(options, redactText);
const context = {
  agentId: "minilda", sessionKey: "agent:minilda:main",
  sessionId: "synthetic-window-a", runId: "synthetic-turn-a", trigger: "user"
};
"""


def run_javascript(body):
    """Execute real local helpers on a fixture and return their JSON output."""
    result = subprocess.run(
        [NODE, '--input-type=module', '-e', JS_IMPORTS + JS_FIXTURE + body],
        cwd=REPO, text=True, capture_output=True, check=True, timeout=20,
    )
    return json.loads(result.stdout)


def show_rows(rows):
    """Display a small result table without a dataframe dependency."""
    columns = list(rows[0])
    header = ''.join(f'<th>{html.escape(key)}</th>' for key in columns)
    body = ''.join('<tr>' + ''.join(
        f'<td>{html.escape(str(row[key]))}</td>' for key in columns
    ) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))

## 3. How the pieces fit together

| File | Responsibility |
| --- | --- |
| `observer/index.js` | Register passive hooks; sanitize selected records; capture the current prompt with its run/session identity. |
| `observer/live-trace.js` | Match information by exact run ID and build additional attributes for the native exporter. |
| `observer/patch-native-exporter.py` | Add narrow calls to those helpers inside one verified version of the installed exporter. |
| `observer/openclaw.plugin.json` | Allow the explicit scope, live-export and reasoning options. |

The native exporter still owns trace IDs, existing model/tool spans, parent relationships, timings and
transmission to Phoenix. The bridge enriches those records. It does not send a second copy of each run.
The observer's separate JSONL log remains useful for local evidence, but it is not a second live Phoenix trace.

The deployed plugin settings were `sessionScope: "all"`, `liveExport: true` and `captureReasoning: "text"`.
The plugin was allowed conversation access, with prompt injection disabled. Native content capture also had
to permit the input/output fields. In this implementation, "all" means **all sessions for the configured
assistant ID `minilda`**, not every agent on the server. Developers adapting this code must review both
`resolveScope` and `findRun`; the agent ID is currently hard-coded.

The shared in-memory map is keyed by **run ID**, so two turns do not borrow each other's question or answer.
It is shared through `globalThis[Symbol.for(...)]` because the plugin loader and native exporter can import
different module instances. The map is bounded to 256 entries and expires inactive entries after one hour.

In [4]:
scope = run_javascript("""
const event = {prompt: "A synthetic question"};
const other = {...context, agentId: "other", sessionKey: "agent:other:main"};
console.log(JSON.stringify([
  {setting: "Default evaluation-only scope", captured: buildRecord("llm_input", event, context) !== null},
  {setting: "All sessions for this assistant", captured: buildRecord("llm_input", event, context, undefined, options) !== null},
  {setting: "Another assistant", captured: buildRecord("llm_input", event, other, undefined, options) !== null}
]));
""")
assert [row['captured'] for row in scope] == [False, True, False]
show_rows(scope)

setting,captured
Default evaluation-only scope,False
All sessions for this assistant,True
Another assistant,False


## 4. Edit: attach a real conversation identity

The OpenClaw routing key can be reused across conversation resets. Grouping by that key alone could join
unrelated days into one Phoenix session. We use the actual **session window ID** instead, hash it, and
prefix it with the interaction type. The hash is shortened to 24 hexadecimal characters.

This helper uses the native heartbeat trigger or heartbeat key suffix, the cron marker, and the explicit
evaluation prefix to distinguish **heartbeat**, **cron**, **evaluation** and **conversation**.
When no window ID is available it uses the run ID, marks `run_fallback`, and keeps that run separate.
It does not guess a conversation from timestamps.

From `observer/live-trace.js` — `buildSessionAttributes`:

```javascript
export function buildSessionAttributes(event) {
  const run = findRun(event);
  if (!run) return {};
  const kind = run.trigger === "heartbeat" || run.sessionKey.endsWith(":heartbeat") ? "heartbeat"
    : run.sessionKey.includes(":cron:") ? "cron"
    : run.sessionKey.startsWith("agent:minilda:phoenix-eval-") ? "evaluation" : "conversation";
  const identity = run.sessionId ?? run.runId;
  const id = createHash("sha256").update(identity).digest("hex").slice(0, 24);
  return { "session.id": `openclaw:${kind}:${id}`, "openclaw.presentation.version": VERSION,
    "openclaw.presentation.kind": kind, "openclaw.presentation.session_source": run.sessionId ? "session_window" : "run_fallback" };
}
```

In [5]:
sessions = run_javascript("""
const examples = [
  ["First question", context],
  ["Follow-up, same window", {...context, runId: "synthetic-turn-b"}],
  ["New window, same routing key", {...context, runId: "synthetic-turn-c", sessionId: "synthetic-window-b"}],
  ["Isolated heartbeat 1", {...context, runId: "heartbeat-1", trigger: "heartbeat", sessionId: "heartbeat-window-1"}],
  ["Isolated heartbeat 2", {...context, runId: "heartbeat-2", trigger: "heartbeat", sessionId: "heartbeat-window-2"}],
  ["No window available", {...context, runId: "fallback-1", sessionId: undefined}]
];
const rows = examples.map(([example, event]) => {
  const attrs = buildSessionAttributes(event);
  return {example, session: attrs["session.id"], source: attrs["openclaw.presentation.session_source"]};
});
assert.equal(rows[0].session, rows[1].session);
assert.equal(new Set(rows.slice(1).map(row => row.session)).size, 5);
console.log(JSON.stringify(rows));
""")
show_rows(sessions)

example,session,source
First question,openclaw:conversation:9601878a6564ee8787723fc8,session_window
"Follow-up, same window",openclaw:conversation:9601878a6564ee8787723fc8,session_window
"New window, same routing key",openclaw:conversation:bca240fed0e6690379a501a6,session_window
Isolated heartbeat 1,openclaw:heartbeat:f69a37765a4f65edddfb03e7,session_window
Isolated heartbeat 2,openclaw:heartbeat:cb88e0893d770c05a361be6d,session_window
No window available,openclaw:conversation:243b1dcadc95bfb06581495d,run_fallback


**Read the result:** the first two turns share a session but remain different runs. Resetting the window,
starting a fresh isolated heartbeat, or lacking a window ID produces a different group. The hash is an
opaque display identifier, not encryption or anonymization of the captured messages.

## 5. Edit: show the current question and the response at the root

The `llm_input` hook includes both a current `prompt` and accumulated `historyMessages`. We save the first
current prompt seen for the exact run ID. We do not substitute all the history as the question.
The native provider response supplies the visible answer; it is tracked separately from reasoning and tool calls.

From `observer/live-trace.js` — `captureLiveRecord`:

```javascript
export function captureLiveRecord(record) {
  const run = findRun(record?.context);
  if (run && record.hook === "llm_input" && run.prompt === undefined) run.prompt = record.event.prompt;
}
```

From `observer/live-trace.js` — `buildRunAttributes`:

```javascript
export function buildRunAttributes(event, policy) {
  const run = findRun(event);
  if (!run) return { attrs: {} };
  const attrs = { ...buildSessionAttributes(event), "openinference.span.kind": "AGENT",
    "openclaw.presentation.delivery": "unknown", "openclaw.presentation.output_source": "last_provider_response",
    "openclaw.presentation.original_name": "openclaw.harness.run" };
  if (policy.inputMessages) {
    attrs["input.value"] = run.prompt ?? "[Current prompt unavailable]";
    attrs["input.mime_type"] = "text/plain";
  }
  if (policy.outputMessages) {
    attrs["output.value"] = run.finalText || "[No final response captured]";
    attrs["output.mime_type"] = "text/plain";
  }
  const name = { heartbeat: "Heartbeat", conversation: "Conversation", cron: "Scheduled task", evaluation: "Evaluation" }[attrs["openclaw.presentation.kind"]];
  return { attrs, name };
}
```

In [6]:
root = run_javascript("""
captureLiveRecord(buildRecord("llm_input", {
  prompt: "Which document should I read first?",
  historyMessages: [{role: "user", content: "An unrelated older question"}]
}, context, undefined, options));
captureModelOutput(context, {outputMessages: [{role: "assistant", content: "Start with the course overview."}]}, {outputMessages: true});
const result = buildRunAttributes(context, {inputMessages: true, outputMessages: true});
assert.equal(result.attrs["input.value"], "Which document should I read first?");
assert.equal(result.attrs["openclaw.presentation.delivery"], "unknown");
console.log(JSON.stringify(result));
""")
show_rows([{'field': key, 'value': root['attrs'][key]} for key in [
    'input.value', 'output.value', 'openinference.span.kind', 'openclaw.presentation.delivery'
]])
print('Readable root name:', root['name'])

field,value
input.value,Which document should I read first?
output.value,Start with the course overview.
openinference.span.kind,AGENT
openclaw.presentation.delivery,unknown


Readable root name: Conversation


### Exact native exporter insertion: completed and failed roots

Both `recordHarnessRunCompleted` and `recordHarnessRunError` now call:

```javascript
const phoenixRoot = buildRunAttributes(evt, runtime.contentCapturePolicy);
Object.assign(spanAttrs, phoenixRoot.attrs);
// Existing native span selection/creation remains here.
setSpanAttrs(span, spanAttrs);
if (phoenixRoot.name) span.updateName(phoenixRoot.name);
```

The original trusted trace/span selection, native timing and error status remain in place. The root
is marked `AGENT`; its original name is saved in `openclaw.presentation.original_name`.
The existing `addRunAttrs` helper also calls `buildSessionAttributes` so the shared session ID is
available on related native spans.

**Important limit:** root Output is the last captured provider response's visible text. It is not a
Telegram receipt and is not an independent check of task completion. If the last response has no visible
text, the root shows `[No final response captured]` instead of reusing an earlier answer.

In [7]:
missing = run_javascript("""
captureModelOutput(context, {outputMessages: [{role: "assistant", content: "An interim answer"}]}, {outputMessages: true});
captureModelOutput(context, undefined, {outputMessages: true});
const result = buildRunAttributes(context, {outputMessages: true}).attrs;
const next = {...context, runId: "another-turn"};
captureLiveRecord(buildRecord("llm_input", {prompt: "A new question"}, next, undefined, options));
const nextRoot = buildRunAttributes(next, {inputMessages: true, outputMessages: true}).attrs;
assert.equal(nextRoot["output.value"], "[No final response captured]");
console.log(JSON.stringify([
  {case: "Last provider output missing", output: result["output.value"]},
  {case: "New turn has no answer yet", output: nextRoot["output.value"]}
]));
""")
assert all(row['output'] == '[No final response captured]' for row in missing)
show_rows(missing)

case,output
Last provider output missing,[No final response captured]
New turn has no answer yet,[No final response captured]


## 6. Edit: expose recorded reasoning from the right model call

OpenClaw's attempt-level `llm_output` hook is not one event per provider call. Copying its reasoning onto
every model span would misattribute the evidence. The live patch instead reads the actual provider response
at the native model-call boundary. `captureModelOutput` recognizes supported reasoning fields and text blocks,
while keeping visible response text separate.

From `observer/live-trace.js` — `captureModelOutput`:

```javascript
export function captureModelOutput(event, content, policy) {
  const run = findRun(event);
  if (!run) return { attrs: {} };
  const attrs = buildSessionAttributes(event);
  if (!policy.outputMessages) return { attrs };
  const messages = Array.isArray(content?.outputMessages) ? content.outputMessages : [];
  const assistant = messages.filter((message) => message?.role === "assistant").at(-1);
  run.finalText = assistant ? state.clean(readVisibleText(assistant)) : undefined;
  const texts = assistant ? readReasoningTexts(assistant) : [];
  const mode = state.options.captureReasoning;
  if (mode === "size" || mode === "text") {
    attrs["openclaw.presentation.reasoning_chars"] = texts.reduce((sum, text) => sum + text.length, 0);
    attrs["openclaw.presentation.reasoning_source"] = "native_provider_response";
  }
  return { attrs, reasoningText: mode === "text" && texts.length ? state.clean(texts.join("\n\n")) : undefined };
}
```

The insertion comes immediately **after** `assignOtelModelContentAttributes`. That upstream function creates
the normalized/redacted export attributes without consuming the original `modelContent` object. The helper
reads that original content alongside the normal export; saying that it executes "before redaction" would
misdescribe the code order. It still respects `contentCapturePolicy.outputMessages`.

When text capture is enabled and reasoning is present, the patch creates a **Recorded reasoning** child
of that native model span. Its duration is zero, its kind is `CHAIN`, and
`openclaw.presentation.observation_only` is true. It adds no token counters.

The examples below use **invented diagnostic text**, not any user's or model's private transcript.

In [8]:
reasoning = run_javascript("""
const content = {outputMessages: [{role: "assistant", reasoning_content: "Check the source. secret=demo-value", content: "Here is the answer."}]};
const original = structuredClone(content);
const rows = [];
for (const [mode, allowed] of [["none", true], ["size", true], ["text", true], ["text", false]]) {
  configureLiveCapture({...options, captureReasoning: mode}, redactText);
  const result = captureModelOutput({...context, runId: `${mode}-${allowed}`}, content, {outputMessages: allowed});
  rows.push({mode, output_capture_allowed: allowed,
    reported_chars: result.attrs["openclaw.presentation.reasoning_chars"] ?? "not captured",
    exported_text: result.reasoningText ?? "not captured"});
}
assert.deepEqual(content, original);
assert.match(rows[2].exported_text, /REDACTED/);
assert.equal(rows[3].exported_text, "not captured");
console.log(JSON.stringify(rows));
""")
show_rows(reasoning)

mode,output_capture_allowed,reported_chars,exported_text
none,True,not captured,not captured
size,True,35,not captured
text,True,35,Check the source. secret=[REDACTED]
text,False,not captured,not captured


The character count describes the available source text before masking; the exported text can be shorter.
Strings are bounded to 16,000 characters with a truncation marker. This is best-effort credential masking,
not removal of all personal data. Provider-returned reasoning is recorded text, not a complete account of
internal computation or proof that the stated reasoning caused an action. An absent child does not prove
that the model did no reasoning.

### Capture limits belong to different layers

| Layer | What is retained | What can be missing |
| --- | --- | --- |
| Observer JSONL (`index.js`) | Selected hook fields; at most 16,000 characters per string before the truncation marker, 100 entries per array/object, and depth 8. | System-prompt text, system/developer history, binary payloads, and unsupported fields are omitted. System-prompt length and hash are recorded instead. |
| Readable root and reasoning additions (`live-trace.js`) | The current question, last visible answer and enabled reasoning text, bounded by the observer's text cleaner. | A long answer or reasoning record can be shortened; missing provider content is not reconstructed. |
| Native model/tool spans | Existing OpenClaw content capture, with the configured native limits. | These fields do **not** all inherit the observer's 16,000-character limit. The native policy remains a separate control. |
| Saved OpenClaw trajectory | Runtime evidence useful for cross-checking the export. | In the observed 10:15 PM run, the 39,740-character system-prompt field was replaced with truncation metadata at a 32,768-character field limit. The local trajectory was incomplete too. |

The last row is a specific observed limit, not a claim about every OpenClaw release or every field.
Matching the **number** of exported messages to the reported count does not prove that all their text or
the full system prompt was captured. The exporter changes made the evidence easier to inspect; they did
not remove these limits. Do not fill a missing source field with text from the model's recorded reasoning.
The saved evidence is `runs/heartbeat-isolation-20260928T222529Z/observation-b9d1bd0f3eb97ddfd310a6a84f010745.json`.


## 7. Later edit: readable tool-result names without changing results

The first session-view deployment preserved tool steps, but the message cards still emphasized an opaque
call ID. The second deployment added `addToolResultNames` after native message normalization. It joins each
normalized result to its **exact tool-call ID**, takes the name from the source record, and writes the
normalized message's `name` field. Phoenix already knows how to display that field.

It does not join by array position, invent a name, replace the result body or rewrite the call ID.
Existing names are preserved; unknown or conflicting mappings are left alone. Native tool span names
remain `openclaw.tool.execution`, so existing analysis filters keep working.

In [9]:
tools = run_javascript("""
const messages = [
  {role: "tool", parts: [{type: "tool_call_response", id: "call-b", response: "Search results"}]},
  {role: "tool", parts: [{type: "tool_call_response", id: "call-a", response: "Saved notes"}]},
  {role: "tool", parts: [{type: "tool_call_response", id: "unknown", response: "Unmatched"}]}
];
const content = {inputMessages: [
  {role: "toolResult", toolCallId: "call-a", toolName: "memory_search"},
  {role: "toolResult", toolCallId: "call-b", toolName: "web_search"}
]};
const attributes = {"gen_ai.input.messages": JSON.stringify(messages), "input.value": JSON.stringify(messages)};
const original = structuredClone(content);
addToolResultNames(attributes, context, content, {inputMessages: true});
const named = JSON.parse(attributes["gen_ai.input.messages"]);
assert.deepEqual(named.map(message => message.parts), messages.map(message => message.parts));
assert.deepEqual(content, original);
assert.deepEqual(named.map(message => message.name), ["web_search", "memory_search", undefined]);
console.log(JSON.stringify(named.map(message => ({
  call_id: message.parts[0].id, name: message.name ?? "not guessed", result: message.parts[0].response
}))));
""")
show_rows(tools)

call_id,name,result
call-b,web_search,Search results
call-a,memory_search,Saved notes
unknown,not guessed,Unmatched


The source names above are intentionally in a different order from the normalized results. The correct
labels demonstrate the ID-based join. Only the outer card gains the tool name; the inner
`Tool Result: <ID>` row remains Phoenix's standard display. No Phoenix frontend fork was required.
See `addToolResultNames` in the linked source and the complete patch in the appendix.

## 8. Edit: prevent the aggregate wrapper from counting the same work again

OpenClaw can emit individual model calls and an aggregate `openclaw.model.usage` span. If both carry standard
LLM accounting fields, a consumer can count the same work twice. We keep the individual model calls as the
source of tokens/cost and retain the aggregate's native `openclaw.tokens.*` fields only as diagnostic counters.
The wrapper is marked `CHAIN`, not another LLM call.

From `observer/live-trace.js` — `markAggregateUsage`:

```javascript
export function markAggregateUsage(attributes, event) {
  const prefix = state.options.sessionScope === "all" ? "agent:minilda:" : "agent:minilda:phoenix-eval-";
  if (!state.options.liveExport || !event?.sessionKey?.startsWith(prefix)) return;
  for (const key of Object.keys(attributes)) {
    if (key.startsWith("llm.token_count.") || key.startsWith("gen_ai.usage.") || key.startsWith("llm.cost.")) delete attributes[key];
  }
  attributes["openinference.span.kind"] = "CHAIN";
  attributes["openclaw.presentation.aggregate_usage"] = true;
}
```

In [10]:
usage = run_javascript("""
const individualCalls = [{"llm.token_count.total": 12}, {"llm.token_count.total": 8}];
const aggregate = {"llm.token_count.total": 20, "gen_ai.usage.input_tokens": 15,
  "llm.cost.total": 0.01, "openclaw.tokens.total": 20, "openinference.span.kind": "LLM"};
const before = individualCalls.reduce((sum, call) => sum + call["llm.token_count.total"], 0) + aggregate["llm.token_count.total"];
markAggregateUsage(aggregate, context);
const after = individualCalls.reduce((sum, call) => sum + call["llm.token_count.total"], 0) + (aggregate["llm.token_count.total"] ?? 0);
assert.equal(after, 20);
assert.equal(aggregate["openclaw.tokens.total"], 20);
assert.equal(aggregate["llm.cost.total"], undefined);
console.log(JSON.stringify({before, after, aggregate_kind: aggregate["openinference.span.kind"], diagnostic_total: aggregate["openclaw.tokens.total"]}));
""")
show_rows([{'measurement': key, 'value': value} for key, value in usage.items()])

measurement,value
before,40
after,20
aggregate_kind,CHAIN
diagnostic_total,20


This fixture demonstrates a duplicate-accounting risk, not a measurement that every past Phoenix total
was doubled. When comparing periods across the deployment, sum individual native model-call spans on both
sides. Do not sum parent and child durations: the root already covers the interaction, and recorded-reasoning
children are zero-duration observations. A displayed cost of $0 alone also does not establish zero compute cost.

This edit changes exported span accounting; upstream metric counters and histograms remain unchanged.

## 9. Edit: patch only the exact exporter version we inspected

The patch builder first checks the entire upstream file's SHA-256, then requires each replacement anchor
to occur exactly once. It imports the bridge and changes the following locations; it does not replace the
native trace creation or transport implementation.

| Native location | Added behavior |
| --- | --- |
| Module imports | Load the five bridge helpers from the installed observer directory. |
| `recordHarnessRunCompleted` | Add root fields and a readable name. |
| `recordHarnessRunError` | Add the same fields while preserving the native error status. |
| `addRunAttrs` | Add the session grouping attributes. |
| Provider content export | Add tool-result names and capture the actual provider response. |
| After model-span attributes | Attach the optional zero-duration reasoning child to that span. |
| Aggregate usage export | Remove duplicate standard LLM counters from the wrapper. |

The checks below load the Python builder without running its CLI. A changed exporter must fail the hash
check. A non-unique replacement anchor must fail too. Neither check writes or imports a native runtime file.

In [11]:
spec = importlib.util.spec_from_file_location('notebook_exporter_patch', REPO / 'observer/patch-native-exporter.py')
patch_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(patch_module)

checks = []
for label, operation in [
    ('Different upstream file rejected', lambda: patch_module.build_patch('// changed exporter', '/unused/observer')),
    ('Ambiguous replacement rejected', lambda: patch_module.replace_once('anchor anchor', 'anchor', 'replacement')),
]:
    try:
        operation()
    except ValueError:
        checks.append({'check': label, 'result': 'passed'})
    else:
        raise AssertionError(label)
show_rows(checks)

check,result
Different upstream file rejected,passed
Ambiguous replacement rejected,passed


### Optional: reproduce the saved deployment artifact byte for byte

The original runtime and patched runtime were preserved in ignored local audit directories. If those files
are present, this cell rebuilds the patch **in memory** with the recorded installed plugin path and compares
it with the saved tool-label deployment. It also checks JavaScript syntax on a temporary file, without
loading OpenClaw. Public clones can skip this one audit check; all synthetic demonstrations still work.

Changing the installed plugin path changes the import line and therefore the resulting hash. A hash match
here proves reproducibility against the saved deployment, not that the remote machine still runs those bytes.

In [12]:
import tempfile

original_path = REPO / 'runs/live-conversation-deploy-20260929T011341Z/native-exporter-original.mjs'
deployed_path = REPO / 'runs/tool-label-deploy-20260929T013525Z/native-exporter-patched.mjs'
if original_path.exists() and deployed_path.exists():
    original = original_path.read_text()
    assert hashlib.sha256(original.encode()).hexdigest() == patch_module.UPSTREAM_SHA256
    rebuilt = patch_module.build_patch(original, '/Users/minilda/.openclaw/custom-plugins/phoenix-eval-observer')
    assert rebuilt == deployed_path.read_text()
    assert hashlib.sha256(rebuilt.encode()).hexdigest() == 'dd51b3406e04f1126cc3d05478ff76b2edb4abe81b97572bbdd4f51c7e603b3f'
    with tempfile.TemporaryDirectory() as directory:
        runtime_path = Path(directory) / 'exporter.mjs'
        runtime_path.write_text(rebuilt)
        subprocess.run([NODE, '--check', str(runtime_path)], capture_output=True, text=True, check=True)
    print('Saved deployment reproduced byte for byte; JavaScript syntax check passed.')
else:
    print('Optional deployment comparison skipped: private audit files are not in this checkout.')

Saved deployment reproduced byte for byte; JavaScript syntax check passed.


## 10. Run the existing regression tests

These tests use local fixtures. They do not start the assistant, upload spans or call a model.
They cover scope, grouping, separate requests in one session, missing answers, content controls,
tool-result labels, reasoning capture, local-record handling and aggregate token accounting.
The notebook examples above make several of those assertions visible as teaching tables.

In [13]:
test_result = subprocess.run(
    [NODE, '--test', 'test-observer.js', 'test-live-trace.js'],
    cwd=REPO / 'observer', capture_output=True, text=True, timeout=60,
)
if test_result.returncode:
    print(test_result.stdout)
    print(test_result.stderr)
    raise RuntimeError('Observer regression tests failed.')
for line in test_result.stdout.splitlines():
    if line.startswith(('# tests ', '# pass ', '# fail ', '# skipped ')) or 'ℹ tests ' in line or 'ℹ pass ' in line or 'ℹ fail ' in line:
        print(line)
print('Observer regression suite passed without model or network calls.')

ℹ tests 22
ℹ pass 22
ℹ fail 0
Observer regression suite passed without model or network calls.


## 11. Exercise: distinguish a follow-up from a reset

Keep the same routing key and change only the run ID. Should Phoenix group the two turns together?
Then also change the actual session window ID. What should happen now?

Before running the solution, predict the two Boolean values. A common mistake is to hash the run ID
for every turn, which prevents conversation grouping; another is to hash only the routing key, which
can join turns across a reset.

In [14]:
exercise = run_javascript("""
const first = buildSessionAttributes(context)["session.id"];
const followup = buildSessionAttributes({...context, runId: "exercise-followup"})["session.id"];
const reset = buildSessionAttributes({...context, runId: "exercise-reset", sessionId: "new-window"})["session.id"];
console.log(JSON.stringify({followup_shares_session: first === followup, reset_shares_session: first === reset}));
""")
assert exercise == {'followup_shares_session': True, 'reset_shares_session': False}
show_rows([exercise])

followup_shares_session,reset_shares_session
True,False


## 12. What we verified live, and what remains outside this notebook

The September 28 (PDT) deployment used a timestamped backup, checked both local and staged tests, waited for an
idle boundary and restarted the gateway. The first deployment passed 20 tests; the tool-label update passed
22. The audit compared assistant behavior settings before and after the first deployment and found them
unchanged; the later tool-label update left the entire configuration hash unchanged.

Two ordinary, non-evaluation-tagged verification turns had **different native trace IDs and the same Phoenix
session ID**. We checked their current questions and responses in Phoenix, and opened the first turn's
tool parameters, result and recorded reasoning. A subsequent check showed the tool name on the result card.
Those runs had delivery disabled and are instrumentation checks, not quality experiment results.

**Later live check, 10:15 PM PDT on September 28:** a naturally scheduled heartbeat used the readable
`Heartbeat` root, displayed its actual current prompt and `NO_REPLY` output, and had its own
`openclaw:heartbeat:...` session ID. Its two model calls had two recorded-reasoning children. This confirms
the presentation on a real heartbeat as well as the earlier test conversation; it does not establish that
silence was the correct decision or that a message was delivered. [Open that native trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/b9d1bd0f3eb97ddfd310a6a84f010745).

**Evidence snapshot — September 29, 12:43:23 AM PDT:** the tools-only MCP task series had nine completed runs in repetition 1, three in repetition 2, and none in repetition 3. All nine first-pass responses have now been reviewed. Completion, capture verification and answer quality are separate checks: nine completed traces are not nine correct answers. Eight first-pass exact-output checks pass; the study-plan difference is explained by the installed redactor, with the original strict equality failure retained in a separate reconciliation. The review and frozen counts are saved in the evidence snapshot and first-pass review. The desktop launch context for later repetitions repairs Reminders access and is a separate environment change. These MCP runs are native traces; **no MCP dataset experiment has been imported yet**. Dataset experiments 18 and 19 still refer to the earlier baseline and final-answer-check comparison.

**Latest completed natural heartbeat — September 29, 12:45 AM PDT:** [open its native trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/d6039012d08afec943e3768a13b42b01). It completed in **124.705 seconds**, with **two model calls, three tool calls and 62,618 tokens** summed from the individual model calls. Calendar and two Reminders checks preceded a generated day schedule and overdue laptop alert. No tool error was observed in this run. The new fresh-information instruction and cleaned scratch are present in the captured input; counts of 2 and 6 messages match the reported counts. The native session has no previous session, and its derived Phoenix session ID, three tool-call IDs and final response match the saved transcript. Native verification · Condition-by-condition comparison.

The [12:15 AM heartbeat](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/b0f648ab4d1d9a1910d75fdabdacebc0) provides a useful failure example alongside it: two model calls and four tool calls, including an invalid `memory_get` path that failed without a retry. Its captured input counts are 2 and 7. Both post-edit runs made live checks and generated an alert, but the 11:45 PM run already queried Reminders and generated the laptop alert **before** the latest instruction was added. The latest edit therefore cannot be credited with the first successful discovery. These are live observations under changing conditions, not matched trials proving improvement.

For both recent runs, the saved transcript's first user item is the placeholder `[OpenClaw heartbeat poll]`. The observer supplies the expanded prompt displayed in Phoenix. Its one `historyMessages` item is injected runtime context, not an earlier conversation. Full system-prompt bytes remain omitted, so shared-context influence cannot be completely reconstructed. A generated answer still does not establish Telegram delivery, and dates in personal reminders are not independently verified external deadlines.

### Screenshots of the deployed presentation

These are actual Phoenix UI captures, not synthetic examples or recreated screens. The new images below show the **12:15 AM heartbeat**, not the later 12:45 AM run. Their capture manifest records the original trace/span IDs, capture time and SHA-256 hashes.



*The result cards now identify `apple_calendar_events`, `minilda_reminders__read_reminders` and `memory_get`. Names make the sequence readable; they do not turn a failed call into a successful one. The memory result here is an error. [Source trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/b0f648ab4d1d9a1910d75fdabdacebc0).* 



*Actual first-pass MCP task trace: the final model input contains arXiv search, download and continuation-read results. The trace supports that these retrieval steps occurred, not that the assistant read the entire paper or answered correctly. This is not an imported MCP dataset comparison. [Source trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/febcaf00459ea2fff29ace93f0ab5a8f).* 



*Actual Phoenix screenshot of the final model call in the earlier answer-check study-plan experiment, not the later MCP pass. [Original native trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/49df1623e7be8ead67674792af24d65f). Input is collapsed in this image; inspect the linked trace for the tool evidence.*



*Dataset 3, experiments 18 and 19: original baseline versus final-answer-check. This pairs saved responses and measurements; it is not a model leaderboard or the MCP comparison. The capture manifest preserves source IDs and image hashes. The main analysis notebook contains the full metrics, response-comparison and heartbeat screenshot sequence. No new MCP comparison screenshot is claimed.*

The later silence wording, temporary quiet-hours extension and nine-task comparison are behavioral
experiments documented in the [cookbook](../COOKBOOK.md). They are separate from the exporter edits
explained here. This notebook remains the code deep dive; its executable examples stay synthetic.


| Evidence | What it establishes |
| --- | --- |
| Synthetic examples and regression tests | Helper behavior for the conditions asserted here. |
| Saved original/patched file comparison | The current patch builder reproduces the recorded deployment. |
| Two-turn API readback and UI inspection | Phoenix grouped the observed live turns and displayed their question/response. |
| Independent delivery record or Telegram observation | Whether a particular response reached the user; a model output alone cannot establish this. |

The initial two-turn verification did not emit an aggregate usage span. The regression test, rather than
that live sample, established the aggregate transformation at deployment. Historical traces do not
automatically acquire sessions: the earlier `openclaw-conversations` project contains explicitly labeled
reconstructions with new IDs. New live traces remain in `openclaw-assistant` with their original IDs.

**Remaining limits:** best-effort hooks can miss events; the in-memory correlation cache expires;
content can be truncated; the bridge does not recover full system prompts or unavailable provider data;
recorded reasoning is not complete internal computation; and displayed output is not proof of delivery or correctness.
The patch is tied to one upstream hash and may be replaced by an OpenClaw update. Review, rebuild and
verify it against the new version rather than bypassing the guard.

For another deployment, follow the [installation and rollback guide](../docs/live-conversation-exporter.md).
This notebook deliberately contains no install/restart/export cell. The developer lesson is to preserve
source identity and accounting while making the evidence easy for a person to inspect.

**Optional next extension:** add an explicitly correlated delivery result to the view. Require the exact
interaction identity; do not infer delivery from nearby timestamps or a successful model call.

### Source and audit map

- [Cookbook](../COOKBOOK.md#1-trace-your-assistant)
- [Observer hooks](../observer/index.js), [live helper code](../observer/live-trace.js), [version-checked patch builder](../observer/patch-native-exporter.py)
- [Grouping/content/accounting tests](../observer/test-live-trace.js), [observer tests](../observer/test-observer.js)
- [Live deployment notes](../docs/live-conversation-exporter.md), [historical reconstruction notes](../docs/conversation-tracing.md)
- Local deployment audits (not included in public clones): `runs/live-conversation-deploy-20260929T011341Z/change.json` and `runs/tool-label-deploy-20260929T013525Z/change.json`.

## Appendix: the complete native exporter diff

This diff was generated from the preserved original and the final tool-label deployment artifact.
It includes both revisions. Only the absolute observer import directory is replaced with
`OPENCLAW_PLUGIN_DIR` for readability; the other code edits are unchanged. It is reference material,
not a portable patch to apply to an arbitrary OpenClaw version.

```diff
--- OpenClaw-2026.9.5/native-exporter.mjs
+++ Phoenix-observer-1.1.1/native-exporter.mjs
@@ -1,2 +1,3 @@
+import { buildSessionAttributes, buildRunAttributes, captureModelOutput, markAggregateUsage, addToolResultNames } from "file:///OPENCLAW_PLUGIN_DIR/live-trace.js";
 import { isValidDiagnosticSpanId, isValidDiagnosticTraceFlags, isValidDiagnosticTraceId, redactSensitiveText } from "../api.js";
 import { ROOT_CONTEXT, SpanKind, SpanStatusCode, TraceFlags, context, createContextKey, createNoopMeter, diag, isSpanContextValid, metrics, propagation, trace } from "@opentelemetry/api";
@@ -1469,4 +1470,6 @@
 		const redactedError = normalizeOtelErrorMessage(privateData.errorMessage);
 		if (redactedError) spanAttrs["openclaw.error"] = redactedError;
+		const phoenixRoot = buildRunAttributes(evt, runtime.contentCapturePolicy);
+		Object.assign(spanAttrs, phoenixRoot.attrs);
 		const trustedTrace = trustedTraceContext(evt, metadata);
 		const trackedSpan = trustedTrace?.spanId ? activeTrustedSpans.get(trustedTrace.spanId) : void 0;
@@ -1476,4 +1479,5 @@
 		});
 		setSpanAttrs(span, spanAttrs);
+		if (phoenixRoot.name) span.updateName(phoenixRoot.name);
 		if (evt.outcome === "error") span.setStatus({
 			code: SpanStatusCode.ERROR,
@@ -1502,4 +1506,6 @@
 			...evt.cleanupFailed ? { "openclaw.harness.cleanup_failed": true } : {}
 		};
+		const phoenixRoot = buildRunAttributes(evt, runtime.contentCapturePolicy);
+		Object.assign(spanAttrs, phoenixRoot.attrs);
 		const trustedTrace = trustedTraceContext(evt, metadata);
 		const trackedSpan = trustedTrace?.spanId ? activeTrustedSpans.get(trustedTrace.spanId) : void 0;
@@ -1509,4 +1515,5 @@
 		});
 		setSpanAttrs(span, spanAttrs);
+		if (phoenixRoot.name) span.updateName(phoenixRoot.name);
 		span.setStatus({
 			code: SpanStatusCode.ERROR,
@@ -1974,4 +1981,7 @@
 		assignModelCallUsageAttrs(spanAttrs, evt);
 		assignOtelModelContentAttributes(spanAttrs, modelContent, contentCapturePolicy);
+		addToolResultNames(spanAttrs, evt, modelContent, contentCapturePolicy);
+		const phoenixView = captureModelOutput(evt, modelContent, contentCapturePolicy);
+		Object.assign(spanAttrs, phoenixView.attrs);
 		const span = takeTrackedTrustedSpan(evt, metadata) ?? spanWithDuration(modelCallSpanName(evt), spanAttrs, evt.durationMs, {
 			kind: modelCallSpanKind(),
@@ -1980,4 +1990,13 @@
 		});
 		setSpanAttrs(span, spanAttrs);
+		if (phoenixView.reasoningText) spanWithDuration("Recorded reasoning", {
+			...phoenixView.attrs,
+			"openinference.span.kind": "CHAIN",
+			"input.value": "Recorded reasoning returned by this model call; not a complete account of internal computation.",
+			"input.mime_type": "text/plain",
+			"output.value": phoenixView.reasoningText,
+			"output.mime_type": "text/plain",
+			"openclaw.presentation.observation_only": true
+		}, 0, { parentContext: trace.setSpan(ROOT_CONTEXT, span), endTimeMs: evt.ts }).end(evt.ts);
 		addUpstreamRequestIdSpanEvent(span, evt.upstreamRequestIdHash);
 		if (evt.type === "model.call.error") span.setStatus({
@@ -2504,4 +2523,5 @@
 		assignPositiveNumberAttr(spanAttrs, "gen_ai.usage.cache_read.input_tokens", usage.cacheRead);
 		assignPositiveNumberAttr(spanAttrs, "gen_ai.usage.cache_creation.input_tokens", usage.cacheWrite);
+		markAggregateUsage(spanAttrs, evt);
 		spanWithDuration("openclaw.model.usage", spanAttrs, evt.durationMs, {
 			parentContext: activeTrustedParentContext(evt, metadata),
@@ -2833,4 +2853,5 @@
 	};
 	const addRunAttrs = (spanAttrs, evt) => {
+		Object.assign(spanAttrs, buildSessionAttributes(evt));
 		if (evt.provider) spanAttrs["openclaw.provider"] = evt.provider;
 		if (evt.model) spanAttrs["openclaw.model"] = evt.model;

```

## Recording first-request input size

For the same nine tasks, provider-reported median first-request input was **19,086 tokens in the baseline** and **25,265 with MCP tools**, with 29 versus 41 tool definitions. Both arms had a configured 262,144-token context budget. Record these counts beside correctness, runtime and tool use; the increase does not establish a performance effect or demonstrate context-limit pressure. Counts cover the whole request, and full system prompts are not exported, so they cannot be attributed entirely to tool schemas.

The per-case measurements and offline analysis retain the detailed evidence. The queued model comparison preserves the same tool catalog and records input tokens for each attempt. No exporter behavior changed for this analysis.


## Reading a failed run correctly

A verified Nemotron study-plan attempt on September 29 adds a useful failure example. [Open its native Phoenix trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/9e36dc1f82fc1f2535fd625236792d02). The Evaluation root has status `UNSET`; its `openclaw.run` and second `openclaw.model.call` children have status `ERROR`. The runner records a timeout and no usable final plan. A root row or the existence of an output event alone is therefore insufficient to count task completion.

The exported trace was matched through the native session UUID and the hashed `session.id`. Its observer-context trace ID differed from the exported root trace ID; an empty query for the former did not mean the trace was missing. Preserve both identifiers with their provenance. Native interaction time was 603.221 seconds, actor budget time 600.002 seconds, and setup-inclusive trial time 620.521 seconds.

The first completed call reports 29,431 input and 147 output tokens. The aborted second call reports zeros, which are placeholders rather than proof of zero work. Mark total usage incomplete instead of treating that partial sum as the whole run. Recorded reasoning children remain zero-duration observations; they do not add execution work.

Source accounting · Session mapping. This section explains recorded evidence; it makes no new exporter or assistant changes.

## Preserve JSONL record boundaries

A completed Nemotron response exposed a reader bug in the evaluation supervisor and collector. Tool text contained literal U+2028 characters inside a valid JSON string. Python `str.splitlines()` treated them as extra records and raised an unterminated-string error. JSONL records are separated by LF; use `raw.decode().split("\n")` with blank-line filtering, or iterate physical file lines. Keep invalid JSON errors visible—do not drop malformed records or strip tool text.

The source file was intact: 53 physical records, including the final response. Correct record framing recovered the exact output and original Phoenix session, with 13 model calls and 12 tool calls. The actor was not rerun. Synthetic checks covered U+2028, U+2029 and U+0085, and still rejected invalid JSON. This is an evidence-reader repair, not an exporter behavior or model-instruction change.

Source diagnosis · [Native trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/ffce9fac120a199acd3b7e419ed5ba83).

## Preserve failed attempts when importing experiment results

Nemotron's first complete pass is now Phoenix experiment **23**: eight saved responses and one model timeout, with nine original native trace/session mappings and 90 read-back annotations. [Compare with the three Qwen MCP passes](https://app.phoenix.arize.com/s/matildaorona/datasets/RGF0YXNldDoz/compare?experimentId=RXhwZXJpbWVudDoyMA%3D%3D&experimentId=RXhwZXJpbWVudDoyMQ%3D%3D&experimentId=RXhwZXJpbWVudDoyMg%3D%3D&experimentId=RXhwZXJpbWVudDoyMw%3D%3D&view=grid). This is a saved-result import with no new actor or judge calls.

A timed-out run still has model/tool evidence. Its `task_outcome` is `TIMEOUT` with CODE provenance; completed-answer PASS/FAIL/UNSURE reviews have LLM provenance and explicitly identify the existing assistant review. The captured `[No final response captured]` placeholder stays marked as a capture placeholder, never an actor answer. Native runtime and call counts include that attempt; its full token totals remain `UNKNOWN` without a numeric score. Its first completed call reports 29,578 tokens, while the aborted second stream's usage is unavailable.

This changes how a comparison panel is read. Nemotron's custom `native_seconds` and call averages cover **nine runs**, but `total_tokens` averages cover **eight complete-usage responses**. Phoenix's built-in trace token total can include the timeout's known partial usage; it is not a complete total of consumed tokens. Use the separately recomputed **seven common completed cases** for equal-case runtime comparisons across all three Qwen passes and this first Nemotron pass.

Importer and readback checks · Verified import receipt · Saved metrics and source hashes.



*Analysis figure recomputed from saved native traces and existing fixed-requirement reviews; not a Phoenix screenshot. Setup failure, timeout and uncertainty are retained.*


## Follow an experiment result back to the actual tool request

Phoenix experiment **24**, Nemotron MCP pass 2, contains nine completed responses with their original native trace IDs and 90 verified annotations. Saved import and readback. All nine have complete individual-call usage in this pass; the earlier timeout's unknown usage remains unchanged.

Open the flight row in the experiment comparison, choose **View run trace**, and select the first `openclaw.model.call`. The input header confirms `nemotron-3-super` and **41 tool definitions**; its output contains the actual `flights__search-flight` request for the requested dates. This verifies tool availability and selection separately from the final response's correctness.



The adjacent `openclaw.tool.execution` preserves the returned payload. Comparing that evidence with the answer revealed a mismatched return itinerary in one quoted offer. Both flight answers in the pictured Qwen–Nemotron comparison retain FAIL labels. A useful recommendation can coexist with a specific factual defect.

[Original native trace](https://app.phoenix.arize.com/s/matildaorona/projects/UHJvamVjdDoyMw==/traces/663bfd7863e389b741596889d9f881f6?timeRangeKey=7d&selectedSpanNodeId=U3BhbjoxODQzMQ%3D%3D) · Tool-result capture and screenshot provenance · All five arms with matched runtime denominators.


## Keep timeout accounting and comparison denominators explicit

**Verified snapshot: September 29, 2026, 12:33 UTC.** Phoenix experiment **25**, Nemotron MCP pass 3, preserves all nine planned outcomes: **seven completed answers and two timeouts**. The completed answers have existing assistant-authored reviews of **2 PASS, 3 FAIL and 2 UNSURE**; the two timeouts retain CODE provenance. The saved-result import verified nine rows and 90 annotations without new actor or judge calls. [Open the six-arm comparison](https://app.phoenix.arize.com/s/matildaorona/datasets/RGF0YXNldDoz/compare?experimentId=RXhwZXJpbWVudDoyMA%3D%3D&experimentId=RXhwZXJpbWVudDoyMQ%3D%3D&experimentId=RXhwZXJpbWVudDoyMg%3D%3D&experimentId=RXhwZXJpbWVudDoyMw%3D%3D&experimentId=RXhwZXJpbWVudDoyNA%3D%3D&experimentId=RXhwZXJpbWVudDoyNQ%3D%3D&view=grid) · Import receipt.

The two timeout traces demonstrate why capture completeness, execution outcome and answer quality need separate fields:

| Third-pass case | Actor budget | Native interaction time | Model / tool calls | Known partial tokens | Full token total |
| --- | ---: | ---: | ---: | ---: | --- |
| Study plan | 600 s | 602.925 s | 4 / 3 | 113,288 | UNKNOWN |
| Event preparation | 600 s | 603.700 s | 23 / 22 | 917,612 | UNKNOWN |

Both preserve `[No final response captured]`, with no captured observer response text. Each final interrupted model span lacks usage; the partial sums above cover only calls with recorded usage. **Missing interrupted-call usage remains UNKNOWN, never zero and never the partial sum relabeled as a whole-run total.** The native root covers a slightly wider interval than the fixed actor budget; setup-inclusive timings are wider again. Keep all three measures named explicitly.

- Study-plan native trace: [`59ea50e171b85266e840558092664617`](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/59ea50e171b85266e840558092664617); interrupted model span `15cb10e190fcbc69`. Canonical accounting.
- Event-preparation native trace: [`f998582f33b2729c26f9bbd9afa66375`](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/f998582f33b2729c26f9bbd9afa66375); interrupted model span `449a6d8a4057afa5`. Canonical accounting.

These original native identities were matched using the real session UUID and its hashed Phoenix `session.id`; an observer-context trace ID must not be substituted for the native interaction ID. The same rule used for readable sessions is essential when recovering failure evidence.

**Compute the runtime comparison subset from the outcomes.** Across three Qwen MCP passes and three Nemotron MCP passes, only **six questions completed in every arm**. The report derives that intersection rather than carrying forward the earlier seven-case subset. It excludes study planning and event preparation because of Nemotron timeouts, and LoRA because Qwen pass 3 failed before model inference. All **54 planned outcomes** remain in the correctness and execution comparison; the six-case subset is used only for like-for-like completed-run runtime comparisons and does not erase failures.

For experiment 25, custom native-runtime and call-count averages have a denominator of **nine attempts**, while full input/output/total-token averages have a denominator of **seven complete-usage runs**. The all-attempt token total stays UNKNOWN. Read those denominators before interpreting the dashboard; the built-in trace total may still show known partial usage.

Recomputed metrics and source hashes · Six-arm report · Importer and readback checks.

This is a snapshot of the six verified arms, not a final result for the remaining model queue. Serving limits, shared load, live-source variation and the recorded execution-context changes still limit causal comparisons. No exporter behavior, actor configuration or tutorial code changed for this appendix.


## Detect missing usage in an otherwise completed run

**Verified publication: September 29, 2026 at 13:02 UTC.** Qwen35 MCP pass 1 is Phoenix experiment **26**: all nine responses completed and matched their native captures, with **1 PASS / 7 FAIL / 1 UNSURE** under the existing assistant-authored requirements. Actual model identity and **41 tool definitions** are verified on every model call. Execution, quality and measurement completeness are independent dimensions.

The native model spans report **zero input/output/total counters on all 42 calls**, despite nonempty requests and responses. The first-two-case audit also found zero counters in the native session records, while observer model-call-end events omitted usage. This is not evidence that the model used no tokens, and it does not establish the exact upstream provider/adapter cause. All nine per-run token totals and first-request input counts remain **UNKNOWN**. Numeric usage coverage is **0/9**, not nine measured zeroes.

| What remains measurable | Qwen35 first pass | Token accounting |
| --- | ---: | --- |
| Completed native interactions | 9 | Full input/output/total usage UNKNOWN |
| Native whole-run seconds, all nine | 892.943 | Do not derive tokens from elapsed time |
| Native model / tool calls | 42 / 67 | Counts do not imply available token usage |
| Selected MCP tool executions | 17 | Tool access is verified independently |
| First-request tool definitions | 41 on each attempt | First-request input tokens UNKNOWN |

### Preserve source links without importing placeholder usage

Experiment 26 imports the exact saved responses with no direct `trace_id` association. Otherwise Phoenix's built-in experiment usage totals would inherit the native zero placeholders. All **90 annotations** retain the original native trace ID, session ID and URL in metadata and an **Original native trace** Markdown link in their explanation payloads. The verified Phoenix UI currently does **not** render that explanation link, and its **View run trace** action is disabled for this unlinked arm. Users can copy `source_native_url` from annotation metadata or follow the clickable native links in the per-case report. This source-navigation tradeoff is explicit; native evidence remains available. The **36 token annotations** have label `UNKNOWN` and no numeric score. Runtime and tool metrics still use nine attempts. The later four-selected-pass Metrics capture shows Qwen35 usage as dashes; the adjacent +0% is a UI placeholder, not measured zero or no change. This is an explicit publication representation; original responses, native spans and raw zero counters are unchanged.

[Native flight trace](https://app.phoenix.arize.com/s/matildaorona/redirects/traces/21e5f82c39b60486d35cb796eac17200) · First-two accounting audit · Native session accounting · Publication readback · Independent exact-output and metric audit.

The final comparison contains **72 planned outcomes and 65 answers**, with **6 model timeouts and 1 setup failure** kept separate. Three Qwen122B passes, three Nemotron passes and one pass each of Qwen35 and Muse contribute denominators 27/27/9/9. Runtime now uses only **3 of 9 task types per pass** completed in every arm: flights, event inquiry and conference preparation. This gives 9/9/3/3 matched observations, with pooled medians **145.8 / 228.6 / 105.5 / 307.1 seconds**. All 24 matched answers are FAIL or UNSURE: the measurement is time to a completed reply, not time to a correct answer. Final report · Metrics and coverage.

### Reconcile unavailable tools and failed capture checks explicitly

Muse adds **6 completed answers and 3 timeouts**, with **1 PASS / 5 FAIL** among the answers. Usage is UNKNOWN across all **72 observed model calls**; captured zero placeholders do not become token scores. The four-pass Metrics view selects Qwen122B pass 1, Nemotron pass 2, Qwen35 pass 1 and Muse pass 1. Both smaller-model columns show unavailable usage; adjacent +0% placeholders do not measure a zero. Both unlinked imports retain disabled direct trace navigation, so the per-case Muse report supplies clickable native sources. Actual UI evidence · Muse publication readback.

The conference trial's frozen configuration expected 41 tools, but the captured model requests contained **40/41 schemas and 11/12 MCP tools** because Kiwi failed to connect during gateway startup. Five requested calls produced three executed native tool spans and two pre-execution validation errors. The original collector's failed catalog/count guards remain unchanged; an exact case/trace/hash-bound reconciliation permits post-run reporting without claiming that all attempts had identical effective capabilities. Other cases' unmeasured attempt/error counters stay unknown, not zero. Reconciliation receipt · Readiness impact.

The research timeout offers a separate diagnostic: six failed abstract calls reused a shortened numeric identifier, a later web fetch recovered the paper, and the response still ended unfinished at the fixed budget. The tool schema required a string; recorded arguments alone do not locate the conversion in the model versus adapter. Preserve the partial draft separately and ungraded. Native timeout accounting · Actual failed-call capture.

No exporter, actor configuration or tutorial code was changed to mask the missing measurement. No model or judge calls were made for this publication, and the notebook's existing 14 code-cell objects remain intact.
